# Phase 2c — Additional Audio Encoders: MERT and MuQ

Revision step 8 from `NEXT_STEPS.md` (PeerJ resubmission): test whether the conclusions depend on CLAP.

| Encoder | Checkpoint (pinned revision) | Layers | Dim | Pre-training |
|---|---|---|---|---|
| MERT | `m-a-p/MERT-v1-95M` @ `12af15fe` | 13 (input + 12) | 768 | Self-supervised music (acoustic + CQT teachers) |
| MuQ | `OpenMuQ/MuQ-large-msd-iter` @ `0562a578` | 13 | 1024 | Self-supervised music (Mel-RVQ targets), MSD |

Both are self-supervised music encoders; unlike CLAP, neither is trained with text. MuQ weights are released under
CC-BY-NC 4.0 (fine for research use). An Essentia model was planned but `essentia-tensorflow` has no Windows wheels,
so it is not included.

**Protocol (same windowing as Phase 2b):** load each clip at 24 kHz (both models' native rate), split into fixed
non-overlapping 10-s windows (windows < 5 s dropped), batch the equal-length windows of a track together (no padding), mean-pool every
layer's hidden states over time, then average over windows. All layers are kept, because the best layer for
emotion is usually an intermediate one; the layer is chosen on the validation split in Phase 5e, never on test.

**Output:** `outputs/encoders/audio_mert.npy` (N × 13 × 768) and `outputs/encoders/audio_muq.npy` (N × 13 × 1024),
rows in the order of `outputs/encoders/audio_song_ids.csv`. Checkpoints every 50 tracks; safe to re-run.

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import librosa
import torch
from tqdm import tqdm

warnings.filterwarnings('ignore')
torch.set_num_threads(8)

ROOT = os.environ.get('MUSIC_RESEARCH_ROOT', os.path.abspath('.'))
OUT_DIR = os.path.join(ROOT, 'outputs')
ENC_DIR = os.path.join(OUT_DIR, 'encoders')
os.makedirs(ENC_DIR, exist_ok=True)

SR, WIN, MIN_WIN, DURATION, CKPT_EVERY = 24_000, 240_000, 120_000, 30, 50
MERT_REV = '12af15fef9d0ac838c3f475bfbbf26d2060dd4f5'
MUQ_REV  = '0562a57814f6f8bbd9fdea0a25921a2fce1a841a'

df = pd.read_parquet(os.path.join(OUT_DIR, 'merged_dataset.parquet'))
audio_df = df[df['has_audio']].reset_index(drop=True)
audio_df[['song_id']].to_csv(os.path.join(ENC_DIR, 'audio_song_ids.csv'), index=False)
print(f'{len(audio_df)} tracks')

def windows(y):
    w = [y[i:i + WIN] for i in range(0, len(y), WIN)]
    w = [x for x in w if len(x) >= MIN_WIN]
    return w or [y]

## 1. Generic resumable extraction loop

In [ ]:
def extract(name, embed_fn, n_layers, dim):
    out_path  = os.path.join(ENC_DIR, f'audio_{name}.npy')
    ckpt_path = os.path.join(ENC_DIR, f'audio_{name}_ckpt.npz')
    if os.path.exists(out_path):
        print(f'{name}: already complete'); return np.load(out_path)
    E = np.full((len(audio_df), n_layers, dim), np.nan, dtype=np.float32)
    if os.path.exists(ckpt_path):
        E = np.load(ckpt_path)['E']
    todo = np.where(np.isnan(E[:, 0, 0]))[0]
    print(f'{name}: {len(audio_df) - len(todo)} done, {len(todo)} to go')
    t0 = time.time()
    for j, i in enumerate(tqdm(todo, desc=name, mininterval=60)):
        y = librosa.load(audio_df.audio_path[i], sr=SR, mono=True, duration=DURATION)[0]
        wins = windows(y)
        per_win = []                                        # (n_layers, dim) time-mean per window
        with torch.no_grad():
            for L in sorted({len(w) for w in wins}):        # batch equal-length windows only: no padding
                group = [w for w in wins if len(w) == L]
                hs = embed_fn(group)                        # tuple of n_layers tensors (n_group, T, dim)
                per_win += list(torch.stack([h.mean(1) for h in hs], 1).numpy())
        E[i] = np.mean(per_win, axis=0)
        if (j + 1) % CKPT_EVERY == 0:
            np.savez(ckpt_path, E=E)
    assert not np.isnan(E).any()
    np.save(out_path, E)
    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)
    print(f'{name}: saved {E.shape} in {time.time() - t0:.0f}s')
    return E

## 2. MERT-v1-95M

In [ ]:
from transformers import AutoModel, Wav2Vec2FeatureExtractor
mert = AutoModel.from_pretrained('m-a-p/MERT-v1-95M', trust_remote_code=True, revision=MERT_REV).eval()
mert_proc = Wav2Vec2FeatureExtractor.from_pretrained('m-a-p/MERT-v1-95M', trust_remote_code=True, revision=MERT_REV)
assert mert_proc.sampling_rate == SR

def mert_fn(wins):
    x = mert_proc(wins, sampling_rate=SR, return_tensors='pt')
    return mert(**x, output_hidden_states=True).hidden_states

E_mert = extract('mert', mert_fn, 13, 768)
del mert

## 3. MuQ-large-msd-iter

In [ ]:
from muq import MuQ
muq = MuQ.from_pretrained('OpenMuQ/MuQ-large-msd-iter').eval()

def muq_fn(wins):
    return muq(torch.tensor(np.stack(wins)), output_hidden_states=True).hidden_states

E_muq = extract('muq', muq_fn, 13, 1024)
print('MERT', E_mert.shape, '| MuQ', E_muq.shape)